<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%205/5.1%20What%20Are%20Agents%3F/5.1.1%20Tutorial%20-%20The%20Case%20for%20Structured%20Agents%20%E2%80%93%20Enter%20PydanticAI_OpenRouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pydantic pydantic-ai openai python-dotenv duckduckgo-search rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.8/875.8 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 

## Deep dive into agents!

In this tutorial, we'll explore why structure matters in AI agents and how PydanticAI
provides a robust framework for building reliable, predictable agents.

Think of traditional agents as freestyle conversations - they work, but can be unpredictable.
Structured agents are like having a well-designed form that guides the conversation
toward specific, reliable outcomes.

Learning Objectives:
- Understand the limitations of unstructured agent interactions
- Learn how PydanticAI enforces structure through schemas
- Build a simple structured agent that returns predictable results


In [2]:
import os
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

False

### Understanding the Problem with Unstructured Agents

Let's start by seeing what happens when we use a basic, unstructured approach.
This will help us understand WHY we need structure.


In [7]:
# OpenRouter setup
from getpass import getpass
import os

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4.1-mini"

from openai import OpenAI
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

# PydanticAI model routed through OpenRouter; used by the agents below.
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
openrouter_model = OpenRouterModel(MODEL, provider=OpenRouterProvider(api_key=OPENROUTER_API_KEY))

prompt = "Analyze this product and tell me if I should buy it: iPhone 15 Pro"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}]
)

# The problem: we get unstructured text that's hard to parse
response.choices[0].message.content

Enter your OpenRouter API key: ··········


"To help you decide whether to buy the iPhone 15 Pro, I'll provide an analysis based on its key features, pros, cons, and who it might be best suited for. If you have specific needs or uses in mind, please let me know!\n\n### Key Features of iPhone 15 Pro (general expectations based on recent models and typical Apple upgrades):\n- **Design & Build:** Likely an aerospace-grade titanium frame for durability and lightweight feel.\n- **Display:** Typically a 6.1-inch OLED Super Retina XDR display with ProMotion (120Hz refresh rate) for smooth scrolling and better responsiveness.\n- **Performance:** Powered by Apple’s latest A17 Pro chip, offering top-tier performance and energy efficiency.\n- **Camera System:** Usually a triple-camera setup with improvements in low-light photography, computational photography, and possibly a new periscope telephoto lens for better zoom (up to 5x).\n- **Battery Life:** Typically improved battery efficiency, with moderate to strong all-day battery performanc

### Defining Structure with Pydantic Models

Now let's see how we can define exactly what we want our agent to return.
This is the foundation of structured agents - defining the shape of our data.



In [8]:
class ProductAnalysis(BaseModel):
    """
    This model defines exactly what our agent should return.
    Notice how we're being very specific about data types and descriptions.
    """
    product_name: str = Field(description="The name of the product being analyzed")
    recommendation: str = Field(description="Either 'buy', 'don't buy', or 'maybe'")
    confidence_score: float = Field(description="Confidence from 0.0 to 1.0", ge=0.0, le=1.0)
    key_pros: List[str] = Field(description="Top 3 advantages of this product")
    key_cons: List[str] = Field(description="Top 3 disadvantages of this product")
    reasoning: str = Field(description="Brief explanation of the recommendation")

### Creating Our First Structured Agent

Now we'll combine our schema with PydanticAI to create an agent that
ALWAYS returns data in our specified format.



In [9]:
product_analyzer = Agent(
    openrouter_model,
    output_type=ProductAnalysis,
    system_prompt="""
    You are a product analysis expert. Analyze products objectively and provide
    structured recommendations based on features, price, reviews, and market position.

    Always be honest about limitations and provide balanced viewpoints.
    Your confidence score should reflect the strength of available evidence.
    """
)

### Adding Intelligence to Our Agent

Let's enhance our agent with some basic reasoning capabilities.
We'll add a function that helps our agent access current information.



In [ ]:
@product_analyzer.tool
async def get_product_info(ctx: RunContext[None], product_name: str) -> str:
    """
    Simulated tool that would normally fetch real product data.
    In a real implementation, this might call APIs, scrape websites, etc.
    """
    # Simulated product database - in reality, this would be dynamic
    product_db = {
        "iPhone 15 Pro": {
            "price": "$999",
            "features": ["48MP camera", "A17 Pro chip", "Titanium build", "USB-C"],
            "reviews": "4.5/5 stars average",
            "availability": "In stock"
        },
        "Samsung Galaxy S24": {
            "price": "$899",
            "features": ["200MP camera", "Snapdragon 8 Gen 3", "7 years updates"],
            "reviews": "4.4/5 stars average",
            "availability": "In stock"
        }
    }

    product_info = product_db.get(product_name, {"error": "Product not found"})
    return f"Product info for {product_name}: {product_info}"


### Running Our Structured Agent

Now let's see our structured agent in action and compare it to the unstructured approach.


In [10]:

# Run our structured agent
result = await product_analyzer.run("Should I buy the iPhone 15 Pro?")

In [11]:
print("Structured output:")
print(f"Product: {result.output.product_name}")
print(f"Recommendation: {result.output.recommendation}")
print(f"Confidence: {result.output.confidence_score:.2f}")
print(f"Pros: {', '.join(result.output.key_pros)}")
print(f"Cons: {', '.join(result.output.key_cons)}")
print(f"Reasoning: {result.output.reasoning}")

Structured output:
Product: iPhone 15 Pro
Recommendation: maybe
Confidence: 0.70
Pros: Advanced A17 Pro chip for improved performance, ProMotion display with high refresh rate, Enhanced camera system with better low-light capabilities
Cons: High price compared to other smartphones, Limited significant design changes from previous model, Potential availability issues at launch
Reasoning: The iPhone 15 Pro offers notable improvements in performance and camera features, making it appealing for users seeking high-end capabilities. However, its high price and lack of major design changes may not justify an upgrade for all users. Confidence is moderate due to limited information on long-term user reviews and market reception.
